**1.1 Task 1:** 
You will need to solve a relaxed version of the NYC instance where we do not have the energy 
constraint.  You  can  use  any  algorithm  we  discussed  in  the  lectures.  Note  that  this  is  equivalent  to 
solving the shortest path problem. 

In [ ]:
import json
import heapq

def load_data():
    with open("G.json", "r") as f:
        G = json.load(f)

    with open("Dist.json", "r") as f:
        Dist = json.load(f)

    with open("Cost.json", "r") as f:
        Cost = json.load(f)

    return G, Dist, Cost

def uniform_cost_search(G, Dist, start, goal):
    frontier = [(0, start, [start])]
    visited = set()

    while frontier:
        frontier.sort()   # smallest total_distance first
        total_distance, current_node, path = frontier.pop(0)

        if current_node in visited:
            continue

        visited.add(current_node)

        if current_node == goal:
            return path, total_distance

        for neighbor in G[current_node]:
            if neighbor not in visited:
                edge_key = f"{current_node},{neighbor}"
                frontier.append((total_distance + Dist[edge_key], neighbor, path + [neighbor]))

    return None, float('inf')

def compute_total_energy(path, Cost):
    total_energy = 0
    for i in range(len(path) - 1):
        edge_key = f"{path[i]},{path[i+1]}"
        total_energy += Cost[edge_key]
    return total_energy

def main():
    G, Dist, Cost = load_data()

    start = "1"
    goal = "50"

    path, shortest_distance = uniform_cost_search(G, Dist, start, goal)

    if path is None:
        print("No path found.")
        return

    total_energy = compute_total_energy(path, Cost)

    print("Shortest path:", "->".join(path))
    print("Shortest distance:", shortest_distance)
    print("Total energy cost:", total_energy)


if __name__ == "__main__":
    main()

Shortest path: 1->1363->1358->1357->1356->1276->1273->1277->1269->1267->1268->1284->1283->1282->1255->1253->1260->1259->1249->1246->963->964->962->1002->952->1000->998->994->995->996->987->986->979->980->969->977->989->990->991->2369->2366->2340->2338->2339->2333->2334->2329->2029->2027->2019->2022->2000->1996->1997->1993->1992->1989->1984->2001->1900->1875->1874->1965->1963->1964->1923->1944->1945->1938->1937->1939->1935->1931->1934->1673->1675->1674->1837->1671->1828->1825->1817->1815->1634->1814->1813->1632->1631->1742->1741->1740->1739->1591->1689->1585->1584->1688->1579->1679->1677->104->5680->5418->5431->5425->5424->5422->5413->5412->5411->66->5392->5391->5388->5291->5278->5289->5290->5283->5284->5280->50
Shortest distance: 148648.63722140007
Total energy cost: 294853


**1.1 Task 2**: You will need to implement an uninformed search algorithm (e.g., the DFS, BFS, UCS) to solve 
the NYC instance

In [12]:
import json
import heapq


def load_data():
    with open("G.json", "r") as f:
        G = json.load(f)

    with open("Dist.json", "r") as f:
        Dist = json.load(f)

    with open("Cost.json", "r") as f:
        Cost = json.load(f)

    return G, Dist, Cost

def ucs_with_energy_budget(G, Dist, Cost, start, goal):
    budget=287932
    frontier = [(0, 0, start, [start])]
    visited = {}

    while frontier:
        frontier.sort()  # sort by shortest distance first
        total_distance, total_energy, current_node, path = frontier.pop(0)

        # If we've been here before with less or equal energy and distance, skip
        if current_node in visited:
            prev_dist, prev_energy = visited[current_node]
            if total_distance >= prev_dist and total_energy >= prev_energy:
                continue

        visited[current_node] = (total_distance, total_energy)

        if current_node == goal:
            return path, total_distance, total_energy

        for neighbor in G[current_node]:
            edge_key = f"{current_node},{neighbor}"

            new_distance = total_distance + Dist[edge_key]
            new_energy = total_energy + Cost[edge_key]

            if new_energy <= budget:
                frontier.append((new_distance, new_energy, neighbor, path + [neighbor]))

    return None, float('inf'), float('inf')

def main_with_budget():
    start = "1"
    goal = "50"
    budget = 287932
    G, Dist, Cost = load_data()


    path, shortest_distance, total_energy = ucs_with_energy_budget(G, Dist, Cost, start, goal)

    if path is None:
        print("No feasible path found.")
    else:
        print("Shortest path:", "->".join(path))
        print("Shortest distance:", shortest_distance)
        print("Total energy cost:", total_energy)

if __name__ == "__main__":
    main_with_budget()

Shortest path: 1->1363->1358->1357->1356->1276->1273->1277->1269->1267->1268->1284->1283->1282->1255->1253->1260->1259->1249->1246->963->964->962->1002->952->1000->998->994->995->996->987->986->979->980->969->977->989->990->991->2465->2466->2384->2382->2385->2379->2380->2445->2444->2405->2406->2398->2395->2397->2142->2141->2125->2126->2082->2080->2071->1979->1975->1967->1966->1974->1973->1971->1970->1948->1937->1939->1935->1931->1934->1673->1675->1674->1837->1671->1828->1825->1817->1815->1634->1814->1813->1632->1631->1742->1741->1740->1739->1591->1689->1585->1584->1688->1579->1679->1677->104->5680->5418->5431->5425->5424->5422->5413->5412->5411->66->5392->5391->5388->5291->5278->5289->5290->5283->5284->5280->50
Shortest distance: 150335.55441905273
Total energy cost: 259087


**1.1 Task  3**:  You  will  need  to  develop  an  A*  search  algorithm  to  solve  the  NYC  instance.  The  key  is  to 
develop a suitable heuristic function for the A* search algorithm in this setting

In [13]:
import json
import heapq
import math


def load_data():
    with open("G.json", "r") as f:
        G = json.load(f)

    with open("Dist.json", "r") as f:
        Dist = json.load(f)

    with open("Cost.json", "r") as f:
        Cost = json.load(f)

    with open("Coord.json", "r") as f:
        Coord = json.load(f)

    return G, Dist, Cost, Coord


def heuristic(node, goal, Coord):
    x1, y1 = Coord[node]
    x2, y2 = Coord[goal]
    return math.sqrt((x1 - x2) ** 2 + (y1 - y2) ** 2)


def reconstruct_path(parent, goal):
    path = []
    cur = goal
    while cur is not None:
        path.append(cur)
        cur = parent[cur]
    path.reverse()
    return path


def a_star_with_energy_budget(G, Dist, Cost, Coord, start, goal, budget):
    # heap item: (f, g, energy, node)
    frontier = []
    h_start = heuristic(start, goal, Coord)
    heapq.heappush(frontier, (h_start, 0, 0, start))

    parent = {start: None}

    # best known distance for a state (node, energy_used)
    best_g = {(start, 0): 0}

    # dominance pruning:
    # for each node, store non-dominated (energy, distance) pairs
    best_at_node = {start: [(0, 0)]}

    while frontier:
        f, g, energy, node = heapq.heappop(frontier)

        if node == goal:
            path = reconstruct_path(parent, goal)
            return path, g, energy

        for neighbor in G[node]:
            edge_key = f"{node},{neighbor}"

            new_g = g + Dist[edge_key]
            new_energy = energy + Cost[edge_key]

            if new_energy > budget:
                continue

            # dominance check:
            # if we've already reached this neighbor with
            # <= energy and <= distance, skip this state
            dominated = False
            if neighbor in best_at_node:
                for old_energy, old_g in best_at_node[neighbor]:
                    if old_energy <= new_energy and old_g <= new_g:
                        dominated = True
                        break
            if dominated:
                continue

            # remove states dominated by the new one
            if neighbor not in best_at_node:
                best_at_node[neighbor] = []

            filtered = []
            for old_energy, old_g in best_at_node[neighbor]:
                if not (new_energy <= old_energy and new_g <= old_g):
                    filtered.append((old_energy, old_g))
            filtered.append((new_energy, new_g))
            best_at_node[neighbor] = filtered

            state = (neighbor, new_energy)
            if state not in best_g or new_g < best_g[state]:
                best_g[state] = new_g
                parent[neighbor] = node
                h = heuristic(neighbor, goal, Coord)
                heapq.heappush(frontier, (new_g + h, new_g, new_energy, neighbor))

    return None, float("inf"), float("inf")


def main():
    G, Dist, Cost, Coord = load_data()

    start = "1"
    goal = "50"
    budget = 287932

    path, shortest_distance, total_energy = a_star_with_energy_budget(
        G, Dist, Cost, Coord, start, goal, budget
    )

    if path is None:
        print("No feasible path found.")
    else:
        print("Shortest path:", "->".join(path))
        print("Shortest distance:", shortest_distance)
        print("Total energy cost:", total_energy)


if __name__ == "__main__":
    main()

Shortest path: 1->1363->1358->1357->1356->1276->1273->1277->1269->1241->1240->1235->956->953->955->957->958->959->960->962->1002->952->1000->998->994->995->996->987->988->979->980->969->977->989->990->991->2465->2466->2384->2382->2385->2386->2380->2445->2444->2405->2406->2398->2395->2397->2142->2141->2125->2126->2082->2080->2071->1979->1975->1967->1966->1974->1973->1971->1970->1948->1937->1939->1935->1931->1934->1673->1675->1674->1837->1671->1828->1825->1817->1815->1634->1814->1813->1632->1631->1742->1741->1740->1739->1591->1689->1585->1584->1688->1579->1679->1677->104->5680->5418->5431->5425->5424->5422->5413->5412->5411->66->5392->5391->5388->5291->5278->5289->5290->5283->5284->5280->50
Shortest distance: 150335.55441905273
Total energy cost: 259087


**2.1 Task  1**:  The  environment  model  is  fully  known.  The  transition  dynamics  are  deterministic,  and  the 
discount factor is 𝜸=𝟎.𝟗. You are required to implement two planning algorithms: 
(1) Value Iteration: 
Compute the optimal value function using the Bellman optimality equation and derive the optimal policy. 
(2)  Policy  Iteration:  Start  from  an  initial  policy  and  alternate  between  policy  evaluation  and  policy 
improvement until  convergence.


In [ ]:
# ============================================================
# 2.1 Task 1
# Value Iteration + Policy Iteration for the 5x5 Grid World
# ============================================================

GRID_SIZE = 5
BLOCKS = {(1, 2), (3, 2)}
GOAL_STATE = (4, 4)

ACTIONS = ['U', 'D', 'L', 'R']
GAMMA = 0.9

DELTA = {
    'U': (0, 1),
    'D': (0, -1),
    'L': (-1, 0),
    'R': (1, 0),
}

LEFT_OF = {
    'U': 'L',
    'D': 'R',
    'L': 'D',
    'R': 'U',
}

RIGHT_OF = {
    'U': 'R',
    'D': 'L',
    'L': 'U',
    'R': 'D',
}


def get_states():
    states = []
    for y in range(GRID_SIZE):
        for x in range(GRID_SIZE):
            if (x, y) not in BLOCKS:
                states.append((x, y))
    return states


def move(state, action):
    if state == GOAL_STATE:
        return GOAL_STATE

    x, y = state
    dx, dy = DELTA[action]
    nx, ny = x + dx, y + dy

    if nx < 0 or nx >= GRID_SIZE or ny < 0 or ny >= GRID_SIZE:
        return state
    if (nx, ny) in BLOCKS:
        return state

    return (nx, ny)


def reward(next_state):
    if next_state == GOAL_STATE:
        return 10
    return -1


def get_transitions(state, action):
    if state == GOAL_STATE:
        return [(1.0, GOAL_STATE, 0)]

    return [
        (0.8, move(state, action), reward(move(state, action))),
        (0.1, move(state, LEFT_OF[action]), reward(move(state, LEFT_OF[action]))),
        (0.1, move(state, RIGHT_OF[action]), reward(move(state, RIGHT_OF[action])))
    ]


def compute_q_value(V, state, action):
    total = 0.0
    for prob, next_state, r in get_transitions(state, action):
        total += prob * (r + GAMMA * V[next_state])
    return total


def value_iteration(theta=1e-10):
    states = get_states()
    V = {s: 0.0 for s in states}

    while True:
        delta = 0.0
        new_V = V.copy()

        for s in states:
            if s == GOAL_STATE:
                new_V[s] = 0.0
                continue

            best_value = max(compute_q_value(V, s, a) for a in ACTIONS)
            delta = max(delta, abs(best_value - V[s]))
            new_V[s] = best_value

        V = new_V

        if delta < theta:
            break

    policy = {}
    for s in states:
        if s == GOAL_STATE:
            policy[s] = 'G'
        else:
            best_action = max(ACTIONS, key=lambda a: compute_q_value(V, s, a))
            policy[s] = best_action

    return V, policy


def policy_iteration(theta=1e-10):
    states = get_states()
    V = {s: 0.0 for s in states}
    policy = {}

    for s in states:
        if s == GOAL_STATE:
            policy[s] = 'G'
        else:
            policy[s] = 'U'

    stable = False

    while not stable:
        # Policy Evaluation
        while True:
            delta = 0.0
            new_V = V.copy()

            for s in states:
                if s == GOAL_STATE:
                    new_V[s] = 0.0
                    continue

                action = policy[s]
                value = compute_q_value(V, s, action)
                delta = max(delta, abs(value - V[s]))
                new_V[s] = value

            V = new_V
            if delta < theta:
                break

        # Policy Improvement
        stable = True
        for s in states:
            if s == GOAL_STATE:
                continue

            old_action = policy[s]
            best_action = max(ACTIONS, key=lambda a: compute_q_value(V, s, a))
            policy[s] = best_action

            if best_action != old_action:
                stable = False

    return V, policy


def print_value_grid(V, title):
    print(title)
    print()
    for y in reversed(range(GRID_SIZE)):
        row = []
        for x in range(GRID_SIZE):
            s = (x, y)
            if s in BLOCKS:
                row.append("#####".rjust(8))
            else:
                row.append(f"{V[s]:.2f}".rjust(8))
        print("".join(row))
    print()


def print_policy_grid(policy, title):
    print(title)
    print()
    for y in reversed(range(GRID_SIZE)):
        row = []
        for x in range(GRID_SIZE):
            s = (x, y)
            if s in BLOCKS:
                row.append("#####".rjust(6))
            else:
                row.append(policy[s].rjust(6))
        print("".join(row))
    print()


def compare_policies(p1, p2):
    same = True
    for s in get_states():
        if p1[s] != p2[s]:
            same = False
            break

    if same:
        print("Both policies are the same.")
    else:
        print("Both policies are different.")


def main():
    V_vi, P_vi = value_iteration()
    V_pi, P_pi = policy_iteration()

    print_value_grid(V_vi, "Task 1 - Value Iteration: Value Function")
    print_policy_grid(P_vi, "Task 1 - Value Iteration: Policy")

    print_value_grid(V_pi, "Task 1 - Policy Iteration: Value Function")
    print_policy_grid(P_pi, "Task 1 - Policy Iteration: Policy")

    compare_policies(P_vi, P_pi)


if __name__ == "__main__":
    main()

Task 1 - Value Iteration: Value Function

    2.71    4.65    6.93    9.28    0.00
    1.31    2.71   #####    7.16    9.28
    0.16    1.56    3.22    5.05    6.74
   -0.97    0.04   #####    3.35    4.57
   -1.90   -0.90    0.25    1.68    2.68

Task 1 - Value Iteration: Policy

     R     R     R     R     G
     U     U #####     U     U
     R     R     R     U     U
     U     U #####     U     U
     R     R     R     U     U

Task 1 - Policy Iteration: Value Function

    2.71    4.65    6.93    9.28    0.00
    1.31    2.71   #####    7.16    9.28
    0.16    1.56    3.22    5.05    6.74
   -0.97    0.04   #####    3.35    4.57
   -1.90   -0.90    0.25    1.68    2.68

Task 1 - Policy Iteration: Policy

     R     R     R     R     G
     U     U #####     U     U
     R     R     R     U     U
     U     U #####     U     U
     R     R     R     U     U

Both policies are the same.


**2.1 Task 2**: If the agent does not have access to the transition model, you are required to implement Monte 
Carlo (MC) prediction and control using complete episodes. Specifically, you are required to use an ε-
greedy strategy for policy improvement. Use a fixed ε value of 0.1 throughout training. Estimate state–
action values using sampled returns. Train the agent over multiple episodes starting from the start state

In [18]:
import random
from collections import defaultdict

GRID_SIZE = 5
START = (0, 0)
GOAL = (4, 4)
BLOCKS = {(1, 2), (3, 2)}
ACTIONS = ["U", "D", "L", "R"]

ACTION_TO_DELTA = {
    "U": (0, 1),
    "D": (0, -1),
    "L": (-1, 0),
    "R": (1, 0),
}

GAMMA = 0.9
EPSILON = 0.1
STEP_REWARD = -1
GOAL_REWARD = 10

ALL_STATES = [
    (x, y)
    for x in range(GRID_SIZE)
    for y in range(GRID_SIZE)
    if (x, y) not in BLOCKS
]


def is_valid(state):
    x, y = state
    return 0 <= x < GRID_SIZE and 0 <= y < GRID_SIZE and state not in BLOCKS


def is_terminal(state):
    return state == GOAL


def move(state, action):
    if is_terminal(state):
        return state

    dx, dy = ACTION_TO_DELTA[action]
    next_state = (state[0] + dx, state[1] + dy)
    if not is_valid(next_state):
        return state
    return next_state


def reward(state, action, next_state):
    if next_state == GOAL:
        return GOAL_REWARD
    return STEP_REWARD


def perpendicular_actions(action):
    if action in ("U", "D"):
        return ["L", "R"]
    return ["U", "D"]


def sample_stochastic_step(state, action):
    if is_terminal(state):
        return state, 0

    side1, side2 = perpendicular_actions(action)
    r = random.random()

    if r < 0.8:
        actual_action = action
    elif r < 0.9:
        actual_action = side1
    else:
        actual_action = side2

    next_state = move(state, actual_action)
    return next_state, reward(state, action, next_state)


def argmax_q(Q, state):
    best_action = None
    best_value = float("-inf")
    for a in ACTIONS:
        if Q[(state, a)] > best_value:
            best_value = Q[(state, a)]
            best_action = a
    return best_action


def epsilon_greedy_action(Q, state, epsilon=EPSILON):
    if random.random() < epsilon:
        return random.choice(ACTIONS)
    return argmax_q(Q, state)


def generate_episode(Q, max_steps=200):
    episode = []
    state = START

    for _ in range(max_steps):
        if is_terminal(state):
            break

        action = epsilon_greedy_action(Q, state)
        next_state, r = sample_stochastic_step(state, action)
        episode.append((state, action, r))
        state = next_state

        if is_terminal(state):
            break

    return episode


def monte_carlo_control(num_episodes=10000):
    Q = defaultdict(float)
    returns = defaultdict(list)

    for _ in range(num_episodes):
        episode = generate_episode(Q)

        G = 0
        visited = set()

        for t in reversed(range(len(episode))):
            s, a, r = episode[t]
            G = GAMMA * G + r

            if (s, a) not in visited:
                visited.add((s, a))
                returns[(s, a)].append(G)
                Q[(s, a)] = sum(returns[(s, a)]) / len(returns[(s, a)])

    policy = {}
    V = {}

    for s in ALL_STATES:
        if is_terminal(s):
            continue
        policy[s] = argmax_q(Q, s)
        V[s] = max(Q[(s, a)] for a in ACTIONS)

    V[GOAL] = 0.0
    return Q, V, policy


def print_values(V, title):
    print(f"\n{title}")
    for y in reversed(range(GRID_SIZE)):
        row = []
        for x in range(GRID_SIZE):
            s = (x, y)
            if s in BLOCKS:
                row.append("#####".rjust(8))
            else:
                row.append(f"{V.get(s, 0):6.2f}".rjust(8))
        print(" ".join(row))


def print_policy(policy, title):
    print(f"\n{title}")
    for y in reversed(range(GRID_SIZE)):
        row = []
        for x in range(GRID_SIZE):
            s = (x, y)
            if s in BLOCKS:
                row.append("#####".rjust(6))
            elif s == GOAL:
                row.append(" G ".rjust(6))
            else:
                row.append(policy.get(s, "?").rjust(6))
        print(" ".join(row))


def main():
    Q, V, policy = monte_carlo_control(num_episodes=10000)

    print_values(V, "Task 2 - Monte Carlo: Value Function")
    print_policy(policy, "Task 2 - Monte Carlo: Policy")


if __name__ == "__main__":
    random.seed(42)
    main()


Task 2 - Monte Carlo: Value Function
    1.35     3.51     6.85     9.51     0.00
    1.18     2.92     5.06     7.64     9.92
   -0.13    #####     3.30    #####     7.22
   -2.03    -0.77     0.69     3.05     4.91
   -2.96    -1.77    -0.30     1.15     2.20

Task 2 - Monte Carlo: Policy
     R      R      R      R     G 
     R      R      R      R      U
     U  #####      U  #####      U
     R      R      R      R      U
     U      R      R      U      U


**2.1 Task 3**: *You will implement tabular Q-learning with an ε-greedy exploration strategy, as covered in the 
lecture. Use a fixed ε value of 0.1 throughout training. The agent interacts with the environment without 
knowledge of the transition model in advance. Use a fixed learning rate of 𝛼 =0.1for updating the Q-
values. 

In [17]:
import random
from collections import defaultdict

GRID_SIZE = 5
START = (0, 0)
GOAL = (4, 4)
BLOCKS = {(1, 2), (3, 2)}
ACTIONS = ["U", "D", "L", "R"]

ACTION_TO_DELTA = {
    "U": (0, 1),
    "D": (0, -1),
    "L": (-1, 0),
    "R": (1, 0),
}

GAMMA = 0.9
EPSILON = 0.1
ALPHA = 0.1
STEP_REWARD = -1
GOAL_REWARD = 10

ALL_STATES = [
    (x, y)
    for x in range(GRID_SIZE)
    for y in range(GRID_SIZE)
    if (x, y) not in BLOCKS
]


def is_valid(state):
    x, y = state
    return 0 <= x < GRID_SIZE and 0 <= y < GRID_SIZE and state not in BLOCKS


def is_terminal(state):
    return state == GOAL


def move(state, action):
    if is_terminal(state):
        return state

    dx, dy = ACTION_TO_DELTA[action]
    next_state = (state[0] + dx, state[1] + dy)
    if not is_valid(next_state):
        return state
    return next_state


def reward(state, action, next_state):
    if next_state == GOAL:
        return GOAL_REWARD
    return STEP_REWARD


def perpendicular_actions(action):
    if action in ("U", "D"):
        return ["L", "R"]
    return ["U", "D"]


def sample_stochastic_step(state, action):
    if is_terminal(state):
        return state, 0

    side1, side2 = perpendicular_actions(action)
    r = random.random()

    if r < 0.8:
        actual_action = action
    elif r < 0.9:
        actual_action = side1
    else:
        actual_action = side2

    next_state = move(state, actual_action)
    return next_state, reward(state, action, next_state)


def argmax_q(Q, state):
    best_action = None
    best_value = float("-inf")
    for a in ACTIONS:
        if Q[(state, a)] > best_value:
            best_value = Q[(state, a)]
            best_action = a
    return best_action


def epsilon_greedy_action(Q, state, epsilon=EPSILON):
    if random.random() < epsilon:
        return random.choice(ACTIONS)
    return argmax_q(Q, state)


def q_learning(num_episodes=10000, max_steps=200):
    Q = defaultdict(float)

    for _ in range(num_episodes):
        state = START

        for _ in range(max_steps):
            if is_terminal(state):
                break

            action = epsilon_greedy_action(Q, state)
            next_state, r = sample_stochastic_step(state, action)

            if is_terminal(next_state):
                best_next = 0
            else:
                best_next = max(Q[(next_state, a)] for a in ACTIONS)

            Q[(state, action)] += ALPHA * (
                r + GAMMA * best_next - Q[(state, action)]
            )

            state = next_state

    policy = {}
    V = {}

    for s in ALL_STATES:
        if is_terminal(s):
            continue
        policy[s] = argmax_q(Q, s)
        V[s] = max(Q[(s, a)] for a in ACTIONS)

    V[GOAL] = 0.0
    return Q, V, policy


def print_values(V, title):
    print(f"\n{title}")
    for y in reversed(range(GRID_SIZE)):
        row = []
        for x in range(GRID_SIZE):
            s = (x, y)
            if s in BLOCKS:
                row.append("#####".rjust(8))
            else:
                row.append(f"{V.get(s, 0):6.2f}".rjust(8))
        print(" ".join(row))


def print_policy(policy, title):
    print(f"\n{title}")
    for y in reversed(range(GRID_SIZE)):
        row = []
        for x in range(GRID_SIZE):
            s = (x, y)
            if s in BLOCKS:
                row.append("#####".rjust(6))
            elif s == GOAL:
                row.append(" G ".rjust(6))
            else:
                row.append(policy.get(s, "?").rjust(6))
        print(" ".join(row))


def main():
    Q, V, policy = q_learning(num_episodes=10000)

    print_values(V, "Task 3 - Q-Learning: Value Function")
    print_policy(policy, "Task 3 - Q-Learning: Policy")


if __name__ == "__main__":
    random.seed(42)
    main()


Task 3 - Q-Learning: Value Function
    2.35     4.95     7.13     9.27     0.00
    1.61     3.24     5.18     7.24     9.33
    0.10    #####     3.19    #####     7.22
   -0.99    -0.33     0.86     2.44     5.00
   -2.03    -1.10     0.04     1.31     2.74

Task 3 - Q-Learning: Policy
     R      R      R      R     G 
     R      R      R      R      U
     U  #####      U  #####      U
     U      R      U      R      U
     R      R      R      R      U
